# Comparativo de rendimiento: baseline → S26, y diagnóstico de latencia

**Proyecto:** Traductor LSP → Castellano
**Fecha del informe:** 2026-07-10 · **Última corrida completa:** S26 (2026-07-09) · **Próxima corrida:** S27 (preparada, sin ejecutar)
**Fuente de datos:** `logs/runs.csv`, `notebooks/informe_rendimiento_s26.ipynb`, benchmarks ONNX en vivo

| KPI | Valor |
|---|---|
| F1-test actual (S26) | **0.4098** (objetivo ≥0.35, +6 966% vs S5) |
| HE3 (generalización) | **FALLA** — ΔF1=0.1663, umbral ≤0.15 |
| Latencia p50 (batch=1) | **1.84 ms** (p95=2.58 ms, ONNX CPU) |
| Throughput (batch=32) | **3 073 s/s** — modelo 3.9 MB, 965K parámetros |


## 1. Comparativo baseline vs. actual

### 1.1 Tabla técnica: 15 sprints, baseline (S5) → actual (S26)

| Sprint | Modelo | F1-test | Acc-test | N clases | Lat (ms) | HE3 | ΔF1 | PSI |
|---|---|---|---|---|---|---|---|---|
| S5 | LogReg (baseline) | 0.0058 | 0.0210 | 1086 | 0.1 | — | — | — |
| S9 | LSTM-Bidir S9 | 0.0109 | 0.0262 | 482 | 48.0 | — | — | — |
| S10 | BiLSTM S10 + HPO | 0.0302 | 0.0438 | 482 | 0.9 | — | — | — |
| S13 | BiLSTM S13 (+LSA64) | 0.3696 | 0.3765 | 193 | 0.0 | FALLA | 0.3496 | 0.0729 |
| S14 | BiLSTM S14 (solo-LSP) | 0.1987 | 0.3575 | 139 | 0.9 | FALLA | — | 0.1459 |
| S15 | BiLSTM S15 (+AEC) | 0.1369 | 0.2053 | 255 | 0.9 | PASA | — | 0.0212 |
| S18 | BiLSTM S18 (grupos AEC fix) | 0.3659 | 0.4084 | 101 | 0.8 | FALLA | — | 0.1462 |
| S19 | BiLSTM S19 (aug agresivo) | 0.3357 | 0.3491 | 101 | 0.8 | FALLA | 0.3107 | 0.0877 |
| S20 | BiLSTM S20 (DANN) | 0.2236 | 0.2327 | 101 | 0.8 | FALLA | 0.2118 | 0.0442 |
| S21 | BiLSTM S21 (body-center) | 0.0833 | 0.1175 | 101 | 1.0 | PASA | 0.0787 | 0.0154 |
| S22 | BiLSTM S22 (body-center v2) | 0.3163 | 0.3377 | 101 | 1.1 | FALLA | 0.2917 | 0.1741 |
| S23 | BiLSTM S23 (velocity) | 0.1778 | 0.1049 | 101 | 0.9 | FALLA | 0.1659 | 0.1247 |
| S24 | BiLSTM S24 (hand-only) | 0.2551 | 0.2817 | 101 | 0.9 | FALLA | 0.2354 | 0.1596 |
| S25 | BiLSTM S25 (src-DANN) | 0.2262 | 0.2458 | 101 | 0.9 | FALLA | 0.2144 | 0.0758 |
| S26 ★ | BiLSTM S26 — actual | 0.4098 | 0.4104 | 91 | 1.0 | FALLA | 0.1663 | 0.1523 |

Mejora F1-test baseline→actual: **0.0058 → 0.4098 (+6 966%)**. Ningún sprint ha logrado F1≥0.35 *y* HE3=PASA de forma simultánea — es el objetivo que persigue S27.

![Fig. 1 — Evolución de F1-test por sprint](../data/informe_f1_evolucion.png)

*Fig. 1 — Evolución de F1-test por sprint. Verde = HE3 pasa, rojo = HE3 falla. Línea punteada = objetivo F1≥0.35.*

### 1.2 Percepción del usuario: Top-K accuracy

F1-macro mide exactitud balanceada por clase (criterio técnico). Top-3/Top-5 aproximan la experiencia real: si el sistema sugiere varias señas candidatas, ¿aparece la correcta entre ellas?

| Sprint | F1 (Top-1) | Top-3 | Top-5 |
|---|---|---|---|
| S10 | 0.030 | — | — |
| S13 | 0.370 | 0.530 | 0.582 |
| S15 | 0.137 | 0.356 | 0.406 |
| S18 | 0.366 | 0.556 | 0.605 |
| S19 | 0.336 | 0.439 | 0.463 |
| S22 | 0.316 | 0.432 | 0.503 |
| S26 ★ | 0.410 | 0.494 | 0.588 |

> **Lectura de usuario (S26):** en 1 de cada 2 señas el modelo sugiere la correcta como top-3; en casi 3 de cada 5, aparece entre las top-5. Como asistencia con lista de candidatos (no solo top-1), la utilidad percibida es mayor que el 41% de F1 aislado.

![Fig. 2 — F1 vs Top-3 vs Top-5](../data/informe_topk.png)

*Fig. 2 — F1 (Top-1) vs Top-3 vs Top-5 en los sprints clave de la trayectoria.*

## 2. Informe de latencia

Benchmark de inferencia ONNX Runtime sobre CPU, modelo activo BiLSTM S26 (3.9 MB, 965K parámetros). 1000–2000 corridas por configuración tras warm-up.

### 2.1 Latencia y throughput por batch size (S26)

| Batch | Mean | p50 | p95 | p99 | Max | Throughput |
|---|---|---|---|---|---|---|
| 1 | 1.72 ms | 1.71 ms | 2.25 ms | 2.72 ms | 3.42 ms | 582 s/s |
| 8 | 6.77 ms | 6.49 ms | 7.49 ms | 9.12 ms | 108.46 ms | 1 183 s/s |
| 32 | 10.41 ms | 10.29 ms | 11.27 ms | 11.80 ms | 76.27 ms | 3 073 s/s |

Distribución extendida (2000 corridas, batch=1): **p50=1.84 ms · p95=2.58 ms · p99=3.34 ms · media=1.88 ms (σ=0.495)**. Los picos de `max` en batch=8/32 (76–108 ms) son colas de warm-up del runtime, no representativas de operación estable — **0 errores de inferencia** registrados en 2000+ corridas por configuración.

### 2.2 Comparativo entre modelos (batch=1, 500 corridas)

| Modelo | N clases | Mean | p50 | p95 | Throughput |
|---|---|---|---|---|---|
| S13 | 193 | 1.86 ms | 1.84 ms | 2.47 ms | 538 s/s |
| S19 | 101 | 1.80 ms | 1.82 ms | 2.33 ms | 555 s/s |
| S26 ★ | 91 | 1.80 ms | 1.79 ms | 2.32 ms | 557 s/s |

La latencia es prácticamente insensible al número de clases (193→91, misma arquitectura): el costo dominante es la proyección 150→128 + BiLSTM, no la capa de salida. Los tres modelos están muy por debajo del umbral de 200 ms para tiempo real.

![Fig. 3 — Distribución de latencia y throughput](../data/informe_latencia.png)

*Fig. 3 — Izquierda: histograma de latencia batch=1 con p50/p95/p99. Derecha: throughput vs batch size.*

### 2.3 Optimizaciones de latencia/despliegue probadas

| Etapa | Resultado | Adoptada |
|---|---|---|
| S9 — LSTM puro PyTorch | 48 ms, inaceptable para tiempo real | ✕ |
| S10 — Export ONNX opset 17 | 48 ms → 0.9 ms (53× reducción) | ✓ |
| S10 — hidden 256→128 | F1 cae 15%; descartada | ✕ |
| S10 — n_layers 2→1 | F1 sin cambio, menos sobreajuste | ✓ |
| S13–S26 — proj(150→128) antes de BiLSTM | F1 +2% con menor costo cómputo | ✓ |
| S26 — Temperature scaling (T*=0.924) | ECE 0.140→0.108, latencia sin cambio | ✓ |
| S26 — ONNX CPU (sin GPU/MPS en producción) | p95<3 ms, throughput 1000+ s/s en batch=32 | ✓ |

### 2.4 Errores detectados y resueltos

- **[S9–S10]** Overfitting severo — brecha val/test de 95%.
  → GroupShuffleSplit + KFold estratificado.
- **[S15]** 66% del holdout con F1=0 — letras A,E,L,N,T,X del abecedario, 1 solo señante.
  → Estrategia por_subgrupo: 3 sub-grupos por letra en abecedario.
- **[S26]** Cross-source HISTORIAS_VINETAS_23 — 234 muestras en holdout con F1=0.007.
  → Dataset S17: sub-grupos balanceados dgi156+vineta → validado en S27.
- **[S21]** HE3 pasa trivialmente — F1 global de solo 0.083 (bug de body-centering).
  → Descartado; no se confunde 'pasar la métrica' con resolver el problema.
- **[S20/S25]** DANN empeora F1 — domain-adversarial training, −15% F1 en ambos intentos.
  → Descartado; augmentation agresiva fue más efectiva.
- **[varios]** Crash de MPS (Apple Silicon) durante entrenamiento.
  → PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0

## 3. Causa raíz de la falla HE3 y estado de la solución

HE3 exige generalización a señantes/sesiones no vistos: ΔF1=|F1-test − F1-holdout-por-grupo| ≤0.15, PSI<0.20, KS p>0.05.

| Sprint | F1-test | F1-ghold | ΔF1 | PSI | KS p | HE3 |
|---|---|---|---|---|---|---|
| S10v2 | 0.0091 | 0.0014 | 0.0077 | 0.0225 | 0.0675 | PASA |
| S11 | 0.0237 | 0.0000 | 0.0237 | 0.0825 | 0.0000 | FALLA |
| S13 | 0.3696 | 0.0200 | 0.3496 | 0.0729 | 0.0000 | FALLA |
| S15 | 0.1369 | 0.1200 | 0.0169 | 0.0212 | 0.1200 | PASA |
| S18 | 0.3659 | 0.0500 | 0.3159 | 0.1462 | 0.0000 | FALLA |
| S19 | 0.3357 | 0.0250 | 0.3107 | 0.0877 | 0.0000 | FALLA |
| S21 | 0.0833 | 0.0046 | 0.0787 | 0.0154 | 0.5000 | PASA |
| S26 ★ | 0.4098 | 0.2436 | 0.1663 | 0.1523 | 0.0000 | FALLA |

S26 queda a **ΔF1=0.0163** del umbral (0.1663 vs 0.15 requerido) — el más cerca que ha estado el proyecto de pasar HE3 con F1 útil.

![Fig. 4 — ΔF1 HE3 por sprint](../data/informe_he3.png)

*Fig. 4 — ΔF1 por sprint frente al umbral 0.15 (izq.), F1-test vs F1-holdout (centro), PSI (der.).*

### 3.1 Peores clases en el holdout de S26

| Clase | N holdout | F1 | Fuente | Causa |
|---|---|---|---|---|
| HISTORIAS_VINETAS_23 | 234 | 0.007 | dgi156/vineta | train=dgi156, holdout=vineta (dominios distintos) |
| HISTORIAS_VINETAS_14 | 73 | 0.067 | dgi156/vineta | ídem |
| HISTORIAS_VINETAS_20 | 58 | 0.066 | dgi156/vineta | ídem |
| ORIGINAL | 23 | 0.099 | — | 0 muestras en training |
| TRES | 14 | 0.000 | aec | pocas muestras en holdout (ruido) |
| CÓMO / CONFLICTO / NO / … | 1–6 | 0.000 | aec | pocas muestras en holdout (ruido) |

Las 3 clases `HISTORIAS_VINETAS_*` por sí solas explican **365 de las ~500 muestras problemáticas del holdout (≈73%)**, y son exactamente el patrón de contaminación cross-source: `dgi156` y `vineta` son la misma sesión narrativa capturada dos veces, tratadas como grupos independientes por `GroupShuffleSplit`.

### 3.2 Solución preparada: dataset S17 + train_s27.py

`scripts/build_dataset_s17.py` ya corrigió esto: para toda clase compartida entre `dgi156` y `vineta`, reparte ambas fuentes en 5 sub-grupos balanceados (`balancear_grupos_hv()`), de modo que el holdout deje de recibir una fuente que el modelo nunca vio en training. `data/dataset_s17.npz` está construido (69.7 MB, generado 2026-07-10). `scripts/train_s27.py` apunta a este dataset y declara como objetivo F1-test≥0.35 *y* HE3 PASA.

> ⚠️ **Corrección sobre el notebook fuente:** `notebooks/informe_rendimiento_s26.ipynb` (2026-07-09) da por hecho que S27 está *"ENTRENANDO (estimado ~10 horas MPS)"*. Verificado en el filesystem el 2026-07-10: no existe `checkpoints/bilstm_s27.pt` ni `.onnx`, y no hay entrada `S27` en `logs/runs.csv`. **S27 está preparado pero aún no se ha ejecutado.** Es el paso pendiente inmediato, no un resultado ya obtenido.
>
> Nota adicional: el fix de S17 resuelve la contaminación *cross-source* (dgi156↔vineta), que explica ≈73% de las muestras problemáticas del holdout. No resuelve el problema estructural, distinto, del abecedario de señante único (6 letras A,E,L,N,T,X sin diversidad de señante) — ese requiere grabación nueva o exclusión explícita de la métrica HE3, y sigue pendiente.

## 4. Evidencia — scripts, datos y figuras

Inventario verificado en disco el 2026-07-10, con tamaños de archivo reales.

### 4.1 Scripts del pipeline activo

| Script | Tamaño | Rol |
|---|---|---|
| build_dataset_s16.py | 14.7 KB | Dataset S16: todas las fuentes LSP + por_subgrupo abecedario |
| build_dataset_s17.py | 15.7 KB | Dataset S17: S16 + sub-grupos balanceados HV (fix cross-source) |
| train_s26.py | 27.2 KB | Entrenamiento BiLSTM S26 sobre dataset_s16.npz (91 clases) — completado |
| train_s27.py | 24.3 KB | Entrenamiento BiLSTM S27 sobre dataset_s17.npz (fix HE3) — pendiente de ejecutar |
| train_s19.py | 25.7 KB | Referencia: aug-agresivo, base de S26/S27 |
| catalogo_datasets_lsp.py | 23.8 KB | Catálogo de datasets LSP disponibles y su estado |
| download_lsp_datasets.py | 14.5 KB | Descarga automática de datasets PUCP/GitHub |

### 4.2 Datos y checkpoints

| Archivo | Tamaño | Detalle |
|---|---|---|
| dataset_s16.npz | 68.6 MB | 12 048 muestras, 91 clases, grupos mejorados |
| dataset_s17.npz | 69.7 MB | Igual a S16 + sub-grupos balanceados HV |
| s16_he3_report.json | — | Análisis HE3 pre-entrenamiento, dataset S16 |
| s17_he3_report.json | — | Análisis HE3 pre-entrenamiento, dataset S17 |
| bilstm_s13.onnx | 4.0 MB | Checkpoint sprint 13 |
| bilstm_s19.onnx | 3.9 MB | Checkpoint sprint 19 |
| bilstm_s26.onnx | 3.9 MB | Checkpoint activo — mejor F1 hasta la fecha |
| bilstm_s27.onnx | pendiente | Se genera al ejecutar train_s27.py |

![Fig. 5 — Panel resumen del estado del proyecto](../data/informe_resumen_s26.png)

*Fig. 5 — Panel resumen del estado del proyecto a S26 (trayectoria F1-test completa, con S27 pendiente marcado).*

### 4.3 Fuentes primarias de este informe

- **logs/runs.csv** — historial de 32 corridas de entrenamiento, columnas f1_test/acc_test/latencia_ms/notas.
- **notebooks/informe_rendimiento_s26.ipynb** — notebook fuente, 16 celdas de código, ejecutado sin errores el 2026-07-09; genera las 5 figuras embebidas en este informe.
- **scripts/build_dataset_s17.py**, función `balancear_grupos_hv()` y `analizar_he3()` — lógica del fix cross-source y cálculo de HE3.
- **scripts/catalogo_datasets_lsp.py** — diagnóstico documentado de la causa raíz del abecedario de señante único.
- Benchmarks de latencia ejecutados en vivo sobre `checkpoints/*.onnx` con `onnxruntime`, CPU, 500–2000 corridas por configuración — sin errores de inferencia registrados.

---

*Próxima acción recomendada: ejecutar `scripts/train_s27.py` (~10 h estimadas) para validar si el fix de grupos cross-source de S17 lleva HE3 a PASA.*